# Paraguay — Fire — Collection 1

Pipeline completo: mapas → área stats → gráficos → composição.
Dados **Fire — Collection 1** do [MapBiomas](https://mapbiomas.org/) via [Google Earth Engine](https://earthengine.google.com/).


## Territórios disponíveis

| Grupo | ID | Territórios | Qtd |
|---|---:|---:|---:|
| Country | `country` | paraguay, paraguay_level1 | 2 |
| Departments | `departments` | alto_paraguay, alto_parana, amambay … | 17 |
| Regions | `regions` | paraguay_occidental, paraguay_oriental | 2 |


## Catálogo de produtos

| # | Produto | Descrição | Viz |
|---:|---|---|---|
| 1 | `accumulated_burned` | Accumulated Burned Area | fire |
| 2 | `accumulated_burned_coverage` | Accumulated Burned Coverage | lulc |
| 3 | `annual_burned` | Annual Burned Area | fire |
| 4 | `annual_burned_coverage` | Annual Burned Coverage | lulc |
| 5 | `fire_frequency` | Fire Frequency | frequency_paraguay |
| 6 | `monthly_burned` | Monthly Burned Area | monthly |


## Validação de configuração

Verifica a consistência dos arquivos YAML e batch antes de executar.


In [ ]:
# Validar configuracao YAML e batch
import os
!python -m src.mapbiomas_data.interfaces.cli --validate


In [ ]:
# ============================================================
# SETUP — roda uma vez no inicio da sessao
# ============================================================

import sys, os, subprocess

repo = "gif_factory"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/wallyboy22/gif_factory.git"], check=True)
%cd gif_factory

# Limpar cache bytecode
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null; echo "Cache limpo"

# Garantir fontes
!apt-get install -qq fonts-dejavu-core 2>/dev/null; echo "OK"

!pip install -q earthengine-api pillow pyyaml google-cloud-storage

from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project="mapbiomas-fire-485203")

print("\nSetup concluido.")


### Grupo: Country (country)

**2 territórios × 6 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "paraguay_fire_col1"
group_id = "country"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["paraguay", "paraguay_level1"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 2
n_prod = 6
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


### Grupo: Departments (departments)

**17 territórios × 6 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "paraguay_fire_col1"
group_id = "departments"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["alto_paraguay", "alto_parana", "amambay", "boqueron", "caaguazu", "caazapa", "canindeyu", "central", "concepcion", "cordillera", "guaira", "itapua", "misiones", "neembucu", "paraguari", "presidente_hayes", "san_pedro"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 17
n_prod = 6
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


### Grupo: Regions (regions)

**2 territórios × 6 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "paraguay_fire_col1"
group_id = "regions"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["paraguay_occidental", "paraguay_oriental"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 2
n_prod = 6
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


In [ ]:
# Gerar batch para TODOS os grupos
import json, tempfile, os
collection_ds = "paraguay_fire_col1"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

all_territories = ["paraguay", "paraguay_level1", "alto_paraguay", "alto_parana", "amambay", "boqueron", "caaguazu", "caazapa", "canindeyu", "central", "concepcion", "cordillera", "guaira", "itapua", "misiones", "neembucu", "paraguari", "presidente_hayes", "san_pedro", "paraguay_occidental", "paraguay_oriental"]

items = []
for tid in all_territories:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = len(all_territories)
n_prod = len(product_ids)
print(f"Batch total: {len(items)} combos ({n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


### Executar tudo (todos os grupos)

Sequência completa: pipeline → area stats → charts → compose.


In [ ]:
# Pipeline: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# Area stats: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# Charts: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# Compose: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


In [ ]:
# Sync final: index + Looker CSVs
# Os dados ja foram para o GCS via --gcs em cada etapa.
!python scripts/index/build_index.py --upload
!python scripts/looker/build_looker_csvs_from_gcs.py --dataset paraguay_fire_col1


---
## Links úteis

| Recurso | Link |
|---|---|
| **Looker Studio** | [Abrir dashboard](https://datastudio.google.com/u/0/reporting/179f6b47-8f6e-4f51-abd5-75b7ae018a2b/page/XDzxF) |
| **GitHub** | [github.com/wallyboy22/gif_factory](https://github.com/wallyboy22/gif_factory) |
| **MapBiomas** | [plataforma.brasil.mapbiomas.org](https://plataforma.brasil.mapbiomas.org) |

---

*Gerado por [Fábrica de GIFs — IPAM / MapBiomas](https://github.com/wallyboy22/gif_factory)*
